In [13]:
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.


In [14]:
path

'/kaggle/input/brain-tumor-mri-dataset'

In [15]:
! cp -r /kaggle/input/brain-tumor-mri-dataset /content/Brain_Tumor

In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , models

In [17]:
train_path = r"/content/Brain_Tumor/Training"
test_path =  r"/content/Brain_Tumor/Testing"

## Split The Data

In [18]:
img_size = (224,224)
batch_size = 32
seed = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    image_size=img_size,
    batch_size=batch_size,
    seed=seed)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split= 0.2,
    subset= "validation",
    image_size= img_size,
    batch_size= batch_size,
    seed= seed
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size= img_size,
    batch_size = batch_size
)


Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.


In [19]:
print("Number of classes: ", len(train_ds.class_names))
print("Class Names:", train_ds.class_names)

Number of classes:  4
Class Names: ['glioma', 'meningioma', 'notumor', 'pituitary']


## Autotune

In [20]:
Autotune = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=Autotune)
validation_ds = validation_ds.cache().prefetch(buffer_size=Autotune)
test_ds = test_ds.cache().prefetch(buffer_size=Autotune)

## Build The Model

In [21]:
# Augmenation

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
    layers.RandomRotation(0.1)
])

In [22]:
# Model Layers

model = models.Sequential([
    layers.InputLayer(shape=(224,224,3)),

    augmentation,

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(4, activation='softmax')
])

In [23]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,826,884 (37.49 MB)

 Trainable params: 9,826,884 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
early_Stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

history = model.fit(
    train_ds,
    epochs=50,
    validation_data=validation_ds,
    callbacks=[early_Stopping],
    verbose=1
    )


Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 27s 94ms/step - accuracy: 0.6125 - loss: 0.9148 - val_accuracy: 0.6750 - val_loss: 0.8388
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.7348 - loss: 0.6842 - val_accuracy: 0.8446 - val_loss: 0.4456
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.7783 - loss: 0.5765 - val_accuracy: 0.7563 - val_loss: 0.6114
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.7935 - loss: 0.5271 - val_accuracy: 0.8277 - val_loss: 0.4505
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8103 - loss: 0.4766 - val_accuracy: 0.8723 - val_loss: 0.3208
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.8391 - loss: 0.4141 - val_accuracy: 0.8170 - val_loss: 0.5757
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8453 - loss: 0.3951 - val_accuracy: 0.8679 - val_loss: 0.3664
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8616 - loss: 0.3500 - val_acc

## Evaluation

In [26]:
loss , acc = model.evaluate(test_ds)
print("Accuracy: ", acc)

50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 105ms/step - accuracy: 0.9237 - loss: 0.8440
Accuracy:  0.9237499833106995
